# d1-omni-600M — multimodal decisions on a free T4 (experimental)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/d1-omni-600M-inference-decisions.ipynb)

Run Liquid AI's [`LiquidAI/d1-omni-600M`](https://huggingface.co/LiquidAI/d1-omni-600M) locally with `transformers`:
ask yes/no, multiple-choice, and rating questions about **text + an image** or **text + a speech clip**,
and get probabilities back from **one forward pass with zero output tokens**.

d1-omni-600M is a **587M encoder-based decision model** post-trained from
[LFM2.5-Encoder-350M](https://huggingface.co/LiquidAI/LFM2.5-Encoder-350M), with a SigLIP2 vision tower
and a 17-layer FastConformer audio encoder. It is **not a chat model and does not write text**.
It is the edge / multimodal sibling of [`LiquidAI/d1-3B`](https://huggingface.co/LiquidAI/d1-3B)
(see [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) for the larger text+image model).
This notebook keeps d1-3B as-is and instead shows what the 600M can do that d1-3B does not cover the same
way: **speech decisions**, plus a text+image triage pair.

> **Experimental / research release.** Liquid AI published this as an early research checkpoint that is
> still under active development. The card reports no official speed numbers. On Decision Index 0.2.1 it
> scores 15.95, well below d1-3B (48.57). Treat the answers as a demo of the API, not a quality stand-in
> for d1-3B.

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-omni-600M/blob/main/LICENSE)
(`license: other`, `lfm1.0`). This is **not** Apache 2.0 or MIT. Commercial use is limited to
organizations under a $10M annual-revenue threshold; read the license before any production or paid use.

**Runtime:** pick *Runtime > Change runtime type > T4 GPU*. The T4 has ~15–16 GB of VRAM, **no native
bfloat16**, and **no FlashAttention-2**. The card was trained in float32 and recommends **float16 on
GPU** — it matched float32 top answers on their text / image / audio checks. **Do not use bfloat16**:
the card says it flipped the top answer on 0.8% of text and 1.7% of audio rows. Weights are ~1.2 GB in
float16 (the Hub file is 2.3 GB of float32); peak use on these short demos should stay a few GB.

**Modality pairing (from the card):** a request carries **images or audio, not both** — passing both
raises `ValueError`. `state` is text, JSON, or `None` when the media is the whole state. `audio` is
**one 16 kHz mono clip** (int16 or float), cut at 30 s. Audio was trained on English speaker↔assistant
requests (utterance type, topic, what the speaker wants).

**Links:** [model card](https://huggingface.co/LiquidAI/d1-omni-600M) ·
[Hugging Face blog](https://huggingface.co/blog/LiquidAI/open-d1) ·
[Liquid AI blog](https://www.liquid.ai/blog/d1-open) ·
[Open d1 Arcade](https://huggingface.co/spaces/LiquidAI/system-one-arcade)


In [ ]:
%pip install -q -U "transformers>=5.15"
%pip install -q pillow torchvision soundfile
!apt-get -qq install -y espeak-ng fonts-liberation

# The model card requires transformers>=5.15 and lists pillow, torchvision, and soundfile.
# -U upgrades Colab's preinstalled transformers. The checkpoint ships its own code
# (trust_remote_code) on top of the LFM2.5 encoder classes.
# espeak-ng synthesizes the Demo B voice note; Liberation fonts keep the Demo A invoice readable.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> If the install upgraded transformers, Colab still has the old version imported, and the upgrade only takes
> effect after a restart. The next cell checks the version and stops with a clear message if you skipped this.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.15"), (
    f"transformers {transformers.__version__} is loaded; d1-omni-600M needs 5.15+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on the mem-efficient and math kernels
# in case a future transformers path tries flash SDP. The model’s own encoder does not request FA2.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
# Card-recommended GPU dtype is float16. Do not switch this to bfloat16 on L4/A100:
# the card measured top-answer flips vs float32 on text and audio.
DTYPE = torch.float16

major, minor = torch.cuda.get_device_capability()
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
if major >= 8:
    print("Note: this GPU can run bfloat16, but the d1-omni card says to use float16 anyway.")


## 1. Load d1-omni-600M

The model ships its own code, so it loads with `trust_remote_code=True` (as in the
[model card](https://huggingface.co/LiquidAI/d1-omni-600M#🏃-how-to-use)). The first run downloads
the 2.3 GB float32 checkpoint and casts it to float16.


In [ ]:
from transformers import AutoModel

MODEL_ID = "LiquidAI/d1-omni-600M"

model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, dtype=DTYPE).to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", f"{n_params / 1e6:.1f}M")
print("dtype:", model.dtype)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")


A question has a `type`, `instructions`, and (for `choice` and `score`) `criteria`:

| `type` | `criteria` | answer |
|---|---|---|
| `noul` | optional `{"true": ..., "false": ...}` | `noul`: P(yes) |
| `choice` | `{name: description}` | `choice`, `confidence`, `probabilities` |
| `score` | 2 to 10 levels, lowest first | `score` (expected level), `confidence`, `probabilities`, `legend` |

`model.system_one(state, questions, images=...)` or `audio=...` reads the state once and answers
every question in the same pass. Image and audio answers are the model's softmax as trained
(text-only answers use the per-type temperatures in `config.json`).

With audio, the remote code rewrites choice options to `option_000: ...` and yes/no to plain
`yes` / `no` — the way those questions were trained — then maps answers back to your option names.


In [ ]:
def show(response):
    for name, a in response["answers"].items():
        if a["type"] == "noul":
            print(f"{name:>12}: P(yes) = {a['noul']:.3f}")
        elif a["type"] == "choice":
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a["probabilities"].items())
            print(f"{name:>12}: {a['choice']} ({a['confidence']:.2f})   [{probs}]")
        else:
            level = round(a["score"])
            print(f"{name:>12}: {a['score']:.2f} of {len(a['legend']) - 1} -> {a['legend'][str(level)]!r}")
    print(f"{'usage':>12}: {response['usage']}")


## 2. Demo A — text + image: document triage and a photo defect

d1-3B already answers questions about a photo. Here the **text state and the image travel together**:
inbox context plus an attachment, then warehouse context plus a QC frame. Both images are drawn in
this cell (no extra download, no third-party photo licence).


In [ ]:
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont


def load_font(size, bold=False):
    names = (
        ("LiberationSans-Bold.ttf", "LiberationSans-Regular.ttf")
        if bold
        else ("LiberationSans-Regular.ttf", "LiberationSans-Bold.ttf")
    )
    roots = (
        Path("/usr/share/fonts/truetype/liberation"),
        Path("/usr/share/fonts/truetype/dejavu"),
    )
    dejavu = ("DejaVuSans-Bold.ttf", "DejaVuSans.ttf") if bold else ("DejaVuSans.ttf", "DejaVuSans-Bold.ttf")
    for root, pair in ((roots[0], names), (roots[1], dejavu)):
        for name in pair:
            path = root / name
            if path.exists():
                return ImageFont.truetype(str(path), size)
    return ImageFont.load_default()


def make_overdue_invoice():
    # Readable overdue invoice for document triage, not OCR.
    img = Image.new("RGB", (768, 1024), (248, 246, 240))
    draw = ImageDraw.Draw(img)
    draw.rectangle((0, 0, 768, 88), fill=(28, 56, 92))
    draw.text((36, 22), "ACME SUPPLIES", font=load_font(36, bold=True), fill="white")
    draw.text((36, 112), "INVOICE  #4821", font=load_font(40, bold=True), fill=(28, 56, 92))
    draw.text((36, 172), "Bill to:  Riverside Cafe", font=load_font(26), fill=(30, 30, 30))
    draw.text((36, 214), "Issued:   19 Sep 2026", font=load_font(26), fill=(30, 30, 30))
    draw.text((36, 256), "Due:      3 Oct 2026", font=load_font(26), fill=(30, 30, 30))
    draw.rectangle((36, 320, 732, 420), fill=(190, 28, 28))
    draw.text((56, 348), "STATUS:  OVERDUE", font=load_font(42, bold=True), fill="white")
    draw.rectangle((36, 460, 732, 620), outline=(28, 56, 92), width=3)
    draw.text((56, 480), "Coffee beans, 20 kg          $640.00", font=load_font(24), fill=(30, 30, 30))
    draw.text((56, 524), "Paper cups, 2 000            $280.00", font=load_font(24), fill=(30, 30, 30))
    draw.text((56, 568), "Filters, case                 $320.00", font=load_font(24), fill=(30, 30, 30))
    draw.text((36, 660), "Amount due:  $1,240.00", font=load_font(34, bold=True), fill=(28, 56, 92))
    draw.text((36, 720), "Please remit payment immediately.", font=load_font(24), fill=(80, 20, 20))
    draw.text((36, 960), "accounts@acme.example", font=load_font(20), fill=(90, 90, 90))
    return img


def make_damaged_parcel():
    # Warehouse QC frame: cardboard box with a torn corner and a dark stain.
    img = Image.new("RGB", (640, 480), (186, 176, 160))
    draw = ImageDraw.Draw(img)
    for y in range(0, 480, 14):
        draw.line((0, y, 640, y), fill=(174, 164, 148), width=1)
    draw.rounded_rectangle((80, 60, 560, 420), radius=10, fill=(196, 150, 92), outline=(110, 78, 40), width=5)
    draw.rectangle((80, 220, 560, 258), fill=(228, 206, 128))
    draw.rectangle((320, 86, 540, 176), fill=(244, 238, 226), outline=(40, 40, 40), width=3)
    draw.text((338, 96), "FRAGILE", font=load_font(32, bold=True), fill=(176, 16, 16))
    draw.text((338, 136), "HANDLE WITH CARE", font=load_font(16, bold=True), fill=(40, 40, 40))
    draw.polygon([(80, 60), (200, 60), (80, 190)], fill=(150, 108, 64))
    draw.line([(88, 68), (230, 230)], fill=(36, 18, 8), width=10)
    draw.line([(120, 60), (250, 250)], fill=(24, 12, 6), width=6)
    draw.ellipse((150, 290, 290, 390), fill=(72, 44, 22))
    draw.ellipse((175, 310, 255, 375), fill=(40, 22, 12))
    return img


invoice = make_overdue_invoice()
parcel = make_damaged_parcel()

print("Inbox attachment")
display(invoice.resize((288, 384)))

inbox_state = "Customer emailed this attachment. Route it before the inbox fills up."
inbox_questions = {
    "doc_type": {
        "type": "choice",
        "instructions": "What kind of document is this?",
        "criteria": {
            "invoice": "A bill asking for payment",
            "packing_slip": "A list of items in a shipment",
            "receipt": "Proof a payment already went through",
            "other": "Something else",
        },
    },
    "overdue": {
        "type": "noul",
        "instructions": "Is payment overdue?",
        "criteria": {"true": "The document marks the bill as overdue", "false": "Payment is not overdue"},
    },
    "queue": {
        "type": "choice",
        "instructions": "Which queue should handle this?",
        "criteria": {
            "billing": "Invoices, charges, collections",
            "shipping": "Parcels, tracking, warehouse",
            "support": "Product help or account questions",
        },
    },
    "severity": {
        "type": "score",
        "instructions": "How urgently should a human look at this?",
        "criteria": ["Can wait until later this week", "Handle today", "Collections / blocking the customer"],
    },
}

show(model.system_one(inbox_state, inbox_questions, images=[invoice]))

print("\nWarehouse QC photo")
display(parcel.resize((320, 240)))

qc_state = "Warehouse QC on belt 3. Hold damaged goods before they leave the building."
qc_questions = {
    "damaged": {
        "type": "noul",
        "instructions": "Does the parcel show visible damage?",
        "criteria": {"true": "Tear, crush, stain, or broken corner", "false": "The box looks intact"},
    },
    "action": {
        "type": "choice",
        "instructions": "What should the line do with this parcel?",
        "criteria": {
            "ship": "Looks fine — send it out",
            "inspect": "Possible damage — pull aside for a closer look",
            "reject": "Clearly damaged — do not ship",
        },
    },
    "severity": {
        "type": "score",
        "instructions": "How bad is the damage?",
        "criteria": ["None or cosmetic only", "Needs a human look", "Not shippable"],
    },
}

show(model.system_one(qc_state, qc_questions, images=[parcel]))
print(f"\nVRAM after images: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.max_memory_allocated() / 1e9:.2f} GB peak")


## 3. Demo B — text + audio: voice-note routing

This is the differentiator versus d1-3B. The clip is synthesized with `espeak-ng` (no recorded-voice
licence) as a short English request to an assistant — the setting the card says audio was trained on.
Samples are resampled to **16 kHz mono**, which is what `audio.py` accepts. Clips are cut at 30 s.

Passing `images` and `audio` on the same request raises `ValueError`; this cell sends audio only.


In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, display


def resample_mono(samples, src_rate, dst_rate=16000):
    samples = np.asarray(samples)
    if samples.ndim > 1:
        samples = samples.mean(axis=1)
    if src_rate == dst_rate:
        return samples.astype(np.float32, copy=False)
    n_out = max(1, int(round(len(samples) * dst_rate / src_rate)))
    t_in = np.linspace(0.0, 1.0, num=len(samples), endpoint=False)
    t_out = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
    return np.interp(t_out, t_in, samples.astype(np.float64)).astype(np.float32)


def ensure_espeak():
    if shutil.which("espeak-ng") or shutil.which("espeak"):
        return shutil.which("espeak-ng") or shutil.which("espeak")
    print("Installing espeak-ng …")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
    exe = shutil.which("espeak-ng") or shutil.which("espeak")
    if not exe:
        raise RuntimeError("espeak-ng did not install; cannot synthesize a licence-clean clip.")
    return exe


def synthesize_voice_note(text, path):
    exe = ensure_espeak()
    subprocess.run([exe, "-s", "145", "-v", "en-us", "-w", str(path), text], check=True)
    audio, rate = sf.read(path, dtype="float32")
    return resample_mono(audio, rate, 16000)


spoken = (
    "Hey, can you cancel my flight to Chicago tomorrow and book a later one? "
    "The weather looks really bad."
)
wav_path = Path(tempfile.gettempdir()) / "d1-omni-voice-note.wav"
audio = synthesize_voice_note(spoken, wav_path)
print(f"clip: {len(audio) / 16000:.1f}s @ 16 kHz mono  ({spoken!r})")
display(Audio(audio, rate=16000))

voice_state = "Voice note from a user talking to an assistant."
voice_questions = {
    "topic": {
        "type": "choice",
        "instructions": "What is the speaker talking about?",
        "criteria": {
            "travel": "Travel, flights, or transport",
            "billing": "Charges, invoices, or refunds",
            "food": "Food, groceries, or meals",
            "weather": "Only the weather, with no travel request",
        },
    },
    "intent": {
        "type": "choice",
        "instructions": "What does the speaker want the assistant to do?",
        "criteria": {
            "rebook": "Change or replace an existing booking",
            "cancel_only": "Cancel and stop, with no replacement",
            "status": "Check a reservation or a delay",
            "other": "Something else",
        },
    },
    "refund": {
        "type": "noul",
        "instructions": "Is the speaker asking for a refund?",
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["Can wait", "Handle today", "The trip is tomorrow — act now"],
    },
}

# Do not also pass images= here. The remote code raises ValueError if both media are set.
show(model.system_one(voice_state, voice_questions, audio=audio))
print(f"\nVRAM after audio: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.max_memory_allocated() / 1e9:.2f} GB peak")


## 4. Caveats

- **Experimental / research release.** Liquid AI says d1-omni-600M is an early checkpoint under active
  development. They do not publish inference-speed numbers. Decision quality is far below d1-3B on the
  Decision Index; use [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) when you want
  the stronger text+image model.
- **LFM Open License v1.0** (`license: other`). Not Apache / MIT. Commercial use is limited by a $10M
  annual-revenue threshold; [read the license](https://huggingface.co/LiquidAI/d1-omni-600M/blob/main/LICENSE).
- **Images or audio, not both** on one request. `state` may still be text or JSON beside either modality.
  Audio is one **16 kHz mono** clip, max **30 s**. Audio training was English speaker↔assistant
  (utterance type, topic, what the speaker wants).
- **dtype:** GPU **float16** as the card recommends. Skip **bfloat16** (measured top-answer flips) and
  skip **FlashAttention-2** (not on a T4; this notebook does not enable it).
- **Your own decisions:** swap labels and descriptions; descriptive `criteria` help more than long
  instructions. `system_one_batch` packs many requests, each of which may carry its own image list or clip.
- **More context:** [model card](https://huggingface.co/LiquidAI/d1-omni-600M) ·
  [open-d1 blog](https://huggingface.co/blog/LiquidAI/open-d1) ·
  [Open d1 Arcade](https://huggingface.co/spaces/LiquidAI/system-one-arcade)
